# 🔗 Hands-On Bab 11 — Tantangan AJS
### Analisis Jaringan Sosial: Konsep, Metode, dan Aplikasi

Notebook penutup ini mendemonstrasikan **tantangan metodologis** secara empiris: bagaimana data tidak lengkap dan pemilihan interval waktu dapat mengubah hasil analisis — sekaligus menerapkan **strategi mengatasi tantangan (11.6)** seperti uji sensitivitas dan perbandingan multi-metrik. Diakhiri dengan **Latihan Soal Bab 11**.

In [1]:
!pip install -q networkx matplotlib
import networkx as nx
import matplotlib.pyplot as plt
import random

## 11.2.1 & 11.2.2 — Tantangan Data Tidak Lengkap dan Reliabilitas

Salah satu tantangan metodologis terbesar AJS: hasil metrik jaringan bisa sangat sensitif terhadap data yang hilang. Mari buktikan secara empiris dengan menghapus sebagian simpul secara acak dan mengamati perubahan pada ranking centrality.

In [2]:
G_lengkap = nx.karate_club_graph()
ranking_lengkap = sorted(nx.betweenness_centrality(G_lengkap).items(), key=lambda x: -x[1])
top5_lengkap = [n for n, _ in ranking_lengkap[:5]]
print("Top-5 simpul (data lengkap, 34 simpul):", top5_lengkap)

# Simulasikan data tidak lengkap: hapus 20% simpul secara acak
random.seed(1)
simpul_dihapus = random.sample(list(G_lengkap.nodes()), k=int(0.2 * G_lengkap.number_of_nodes()))
G_tidak_lengkap = G_lengkap.copy()
G_tidak_lengkap.remove_nodes_from(simpul_dihapus)

ranking_tidak_lengkap = sorted(nx.betweenness_centrality(G_tidak_lengkap).items(), key=lambda x: -x[1])
top5_tidak_lengkap = [n for n, _ in ranking_tidak_lengkap[:5]]
print(f"Simpul yang dihapus (20%): {simpul_dihapus}")
print("Top-5 simpul (data tidak lengkap):", top5_tidak_lengkap)

overlap = set(top5_lengkap) & set(top5_tidak_lengkap)
print(f"\nOverlap top-5 sebelum vs sesudah penghapusan data: {len(overlap)}/5 simpul sama")
print("--> Semakin kecil overlap, semakin TIDAK reliabel metrik ini terhadap data tidak lengkap")

Top-5 simpul (data lengkap, 34 simpul): [0, 33, 32, 2, 31]
Simpul yang dihapus (20%): [8, 4, 16, 3, 15, 24]
Top-5 simpul (data tidak lengkap): [0, 33, 2, 32, 31]

Overlap top-5 sebelum vs sesudah penghapusan data: 5/5 simpul sama
--> Semakin kecil overlap, semakin TIDAK reliabel metrik ini terhadap data tidak lengkap


## 11.6 Strategi Mengatasi Tantangan — Uji Sensitivitas

Salah satu strategi dari buku (poin 4): *"Hasil AJS perlu diuji sensitivitasnya dengan memeriksa apakah hasil berubah ketika sebagian data dihapus."* Mari otomatisasi uji ini dengan banyak percobaan acak.

In [3]:
def uji_sensitivitas(G, proporsi_hapus=0.1, n_percobaan=20):
    ranking_asli = set([n for n, _ in sorted(nx.betweenness_centrality(G).items(), key=lambda x: -x[1])[:5]])
    overlaps = []
    for i in range(n_percobaan):
        random.seed(i)
        n_hapus = max(1, int(proporsi_hapus * G.number_of_nodes()))
        dihapus = random.sample(list(G.nodes()), k=n_hapus)
        G_uji = G.copy()
        G_uji.remove_nodes_from(dihapus)
        ranking_baru = set([n for n, _ in sorted(nx.betweenness_centrality(G_uji).items(), key=lambda x: -x[1])[:5]])
        overlaps.append(len(ranking_asli & ranking_baru))
    return overlaps

overlaps = uji_sensitivitas(G_lengkap, proporsi_hapus=0.1, n_percobaan=20)
print(f"Rata-rata overlap top-5 setelah menghapus 10% data (20 percobaan): {sum(overlaps)/len(overlaps):.2f} / 5")
print(f"Rentang: min={min(overlaps)}, max={max(overlaps)}")
print("\nInterpretasi: jika rata-rata overlap tinggi (mendekati 5), hasil analisis relatif STABIL.")
print("Jika rendah, kesimpulan sangat bergantung pada kelengkapan data -> perlu kehati-hatian ekstra.")

Rata-rata overlap top-5 setelah menghapus 10% data (20 percobaan): 4.35 / 5
Rentang: min=2, max=5

Interpretasi: jika rata-rata overlap tinggi (mendekati 5), hasil analisis relatif STABIL.
Jika rendah, kesimpulan sangat bergantung pada kelengkapan data -> perlu kehati-hatian ekstra.


## 11.2.3 Tantangan Pengukuran pada Jaringan Dinamis — Pengaruh Interval Waktu

Buku menjelaskan bahwa interval waktu yang dipilih (harian vs. mingguan, dsb.) dapat menyembunyikan atau melebih-lebihkan perubahan penting. Mari simulasikan efeknya.

In [4]:
# Simulasikan data interaksi harian selama 2 minggu (14 hari)
random.seed(5)
interaksi_harian = []
simpul = ["A","B","C","D","E"]
for hari in range(14):
    n_interaksi = random.randint(2, 5)
    interaksi_harian.append([tuple(random.sample(simpul, 2)) for _ in range(n_interaksi)])

# Snapshot mingguan (agregasi kasar) vs snapshot harian (granular)
G_mingguan = nx.Graph()
for minggu in [interaksi_harian[:7], interaksi_harian[7:]]:
    for hari in minggu:
        G_mingguan.add_edges_from(hari)

print("Snapshot MINGGUAN (2 titik waktu) -> jumlah sisi unik:", G_mingguan.number_of_edges())

# Bandingkan dengan melihat tiap hari secara granular
sisi_per_hari = [len(set(hari)) for hari in interaksi_harian]
print("Snapshot HARIAN (14 titik waktu) -> jumlah sisi unik per hari:", sisi_per_hari)
print("\nInterpretasi: agregasi mingguan menyembunyikan dinamika harian yang sebenarnya berfluktuasi.")
print("Pemilihan interval waktu harus disesuaikan dengan kecepatan perubahan fenomena yang diteliti (lihat 11.2.3).")

Snapshot MINGGUAN (2 titik waktu) -> jumlah sisi unik: 10
Snapshot HARIAN (14 titik waktu) -> jumlah sisi unik per hari: [4, 5, 3, 3, 2, 4, 2, 4, 4, 2, 2, 3, 3, 4]

Interpretasi: agregasi mingguan menyembunyikan dinamika harian yang sebenarnya berfluktuasi.
Pemilihan interval waktu harus disesuaikan dengan kecepatan perubahan fenomena yang diteliti (lihat 11.2.3).


## 11.6 Strategi — Bandingkan Lebih dari Satu Metrik

Strategi lain dari buku (poin 3): gunakan >1 metrik agar kesimpulan lebih kuat jika konsisten.

In [5]:
G_multi = nx.karate_club_graph()
deg = nx.degree_centrality(G_multi)
bet = nx.betweenness_centrality(G_multi)
eig = nx.eigenvector_centrality(G_multi)

top_deg = set([n for n,_ in sorted(deg.items(), key=lambda x:-x[1])[:5]])
top_bet = set([n for n,_ in sorted(bet.items(), key=lambda x:-x[1])[:5]])
top_eig = set([n for n,_ in sorted(eig.items(), key=lambda x:-x[1])[:5]])

konsisten = top_deg & top_bet & top_eig
print(f"Simpul yang konsisten muncul di top-5 ketiga metrik: {konsisten}")
print(f"--> Simpul ini adalah kandidat aktor kunci paling meyakinkan (didukung >1 metrik)")

Simpul yang konsisten muncul di top-5 ketiga metrik: {0, 33, 32, 2}
--> Simpul ini adalah kandidat aktor kunci paling meyakinkan (didukung >1 metrik)


## ✅ Latihan Soal Bab 11

**1.** Jelaskan mengapa kualitas data menjadi salah satu tantangan utama dalam AJS.

**2.** Apa yang dimaksud dengan skalabilitas dalam konteks analisis jaringan? Mengapa hal ini menjadi penting?

**3.** Mengapa jaringan dinamis lebih sulit dianalisis dibandingkan jaringan statis?

**4.** Jelaskan perbedaan antara tantangan teknis dan tantangan etika dalam AJS.

**5.** Mengapa interpretasi hasil analisis jaringan tidak selalu bersifat langsung dan sederhana?

### ✏️ Jawaban Soal 1–5 (tulis di sini)

_1._

_2._

_3._

_4._

_5._

### 🧮 Latihan tambahan
Jalankan `uji_sensitivitas()` di atas dengan `proporsi_hapus` yang berbeda (mis. 0.05, 0.3, 0.5) dan amati bagaimana stabilitas hasil menurun seiring semakin banyak data yang hilang.

In [ ]:
for proporsi in [0.05, 0.2, 0.4]:
    hasil = uji_sensitivitas(G_lengkap, proporsi_hapus=proporsi, n_percobaan=15)
    print(f"Proporsi data dihapus {proporsi*100:.0f}%: rata-rata overlap top-5 = {sum(hasil)/len(hasil):.2f} / 5")

---
### 📚 Referensi
Bab 11 — *Tantangan AJS* (bab penutup), dalam **Analisis Jaringan Sosial: Konsep, Metode, dan Aplikasi**.

🎉 Selamat! Anda telah menyelesaikan seluruh rangkaian hands-on 11 bab — dari konsep dasar jaringan hingga tantangan praktis dalam penerapan AJS.